Goal: fine tune a model that is able to take text from a banking customer and return an intent, once classified, specific tools can be called to handle the actual action. But the first focus is to train a model that can just take

Business Justification: I work in fintech, having a model like this can save a lot of money in customer support


Data: Banking77 data: https://huggingface.co/datasets/PolyAI/banking77

First the imports

In [43]:
import pandas as pd
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from datasets import load_dataset
import os

setup hugging face

In [88]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


lets load the data

In [107]:
dataset = load_dataset("banking77",trust_remote_code=True)
training_set = dataset["train"].select(range(100))
test_set = dataset["test"].select(range(50))

Lets inspect the data by printing 

In [108]:
label_names = dataset["train"].features['label'].names
for data in training_set:
    text = data["text"]
    label = data["label"]
    print(text+" ************ "+label_names[label])

I am still waiting on my card? ************ card_arrival
What can I do if my card still hasn't arrived after 2 weeks? ************ card_arrival
I have been waiting over a week. Is the card still coming? ************ card_arrival
Can I track my card while it is in the process of delivery? ************ card_arrival
How do I know if I will get my card, or if it is lost? ************ card_arrival
When did you send me my new card? ************ card_arrival
Do you have info about the card on delivery? ************ card_arrival
What do I do if I still have not received my new card? ************ card_arrival
Does the package with my card have tracking? ************ card_arrival
I ordered my card but it still isn't here ************ card_arrival
Why has my new card still not come? ************ card_arrival
I still haven't received my card after two weeks, is it lost? ************ card_arrival
Can you track my card for me? ************ card_arrival
Is there a way to track the delivery of my card

Let's prepare the data for training

In [109]:
def messages_for(item):
    message = f"Please provide label for this customer query. Respond with the label, no explanation\n\n{item['text']}."
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": label_names[label]}
    ]

Lets test the messages_for function

In [110]:
messages_for(training_set[0])

[{'role': 'user',
  'content': 'Please provide label for this customer query. Respond with the label, no explanation\n\nI am still waiting on my card?.'},
 {'role': 'assistant', 'content': 'card_arrival'}]

In [111]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [112]:
print(make_jsonl(training_set.select(range(3))))

{"messages": [{"role": "user", "content": "Please provide label for this customer query. Respond with the label, no explanation\n\nI am still waiting on my card?."}, {"role": "assistant", "content": "card_arrival"}]}
{"messages": [{"role": "user", "content": "Please provide label for this customer query. Respond with the label, no explanation\n\nWhat can I do if my card still hasn't arrived after 2 weeks?."}, {"role": "assistant", "content": "card_arrival"}]}
{"messages": [{"role": "user", "content": "Please provide label for this customer query. Respond with the label, no explanation\n\nI have been waiting over a week. Is the card still coming?."}, {"role": "assistant", "content": "card_arrival"}]}


In [113]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [116]:
write_jsonl(training_set, "jsonl/fine_tune_train.jsonl")

In [117]:
write_jsonl(test_set, "jsonl/fine_tune_validation.jsonl")
print('done')

done


In [119]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [120]:
train_file

FileObject(id='file-3q8jaf9xAEBJjxY2rDSzz1', bytes=23475, created_at=1772803762, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [121]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [122]:
validation_file

FileObject(id='file-TjRPvhpWBAgmdc1PvpiBfc', bytes=11804, created_at=1772803767, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

Now, Lets fine tune

In [126]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="customer_query_classifier"
)
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [127]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-hsadYx2dPyGsvCEAdBpys9Rd', created_at=1772803979, level='info', message='Fine-tuning job started', object='fine_tuning.job.event', data=None, type='message'),
 FineTuningJobEvent(id='ftevent-JI81HNKTkXka8NrsdtMoFyH2', created_at=1772803977, level='info', message='Files validated, moving job to queued state', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-EYrIIfLtKqt0chOzBVKNQT2n', created_at=1772803970, level='info', message='Validating training file: file-3q8jaf9xAEBJjxY2rDSzz1 and validation file: file-TjRPvhpWBAgmdc1PvpiBfc', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-YvsUhxRbNC2W5riFH6wPJ1EI', created_at=1772803970, level='info', message='Created fine-tuning job: ftjob-Hi4J3xhApZL7MIlscLV0EDd0', object='fine_tuning.job.event', data={}, type='message')]

Lets Test our fine tuned model

In [146]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:commuinify:customer-query-classifier:DGPiI1yl'

In [132]:
def test_messages_for(item):
    message = f"Please provide label for this customer query. Respond with the label, no explanation\n\n{item['text']}."
    return [
        {"role": "user", "content": message},
    ]

In [144]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [147]:
print(label_names[training_set[0]['label']])
print(gpt_4__1_nano_fine_tuned(training_set[0]))


card_arrival
card_arrival
